[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/multimodalidad/04-video-ia.ipynb)

# Video e IA — Análisis, transcripción y generación

Extracción de fotogramas para análisis con Claude, transcripción con Whisper
y pipeline de resumen de reuniones.

In [ ]:
# pip install anthropic openai google-generativeai
# ffmpeg debe estar instalado en el sistema
import anthropic, os, base64, json, subprocess, time
from pathlib import Path

client = anthropic.Anthropic()
MODELO = 'claude-haiku-4-5-20251001'
print('Cliente listo. Verificando ffmpeg...')

# Verificar que ffmpeg está disponible
try:
    resultado = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
    version = resultado.stdout.split('\n')[0]
    print(f'✅ {version}')
except FileNotFoundError:
    print('❌ ffmpeg no encontrado. Instalar: brew install ffmpeg (macOS) o apt install ffmpeg (Linux)')

## 1. Crear un video de prueba

In [ ]:
# Crear un video de prueba con ffmpeg (10 segundos con texto)
VIDEO_PRUEBA = '/tmp/video_prueba.mp4'

try:
    subprocess.run([
        'ffmpeg', '-y',
        '-f', 'lavfi',
        '-i', 'color=c=blue:size=640x360:duration=10:rate=25',
        '-vf', "drawtext=text='Demo Video IA':fontsize=40:fontcolor=white:x=(w-text_w)/2:y=(h-text_h)/2",
        '-c:v', 'libx264', '-pix_fmt', 'yuv420p',
        VIDEO_PRUEBA, '-loglevel', 'quiet'
    ], check=True)
    size_mb = Path(VIDEO_PRUEBA).stat().st_size / 1024 / 1024
    print(f'✅ Video creado: {VIDEO_PRUEBA} ({size_mb:.2f} MB, 10 segundos)')
except Exception as e:
    print(f'Error creando video: {e}')
    print('Puedes usar cualquier archivo .mp4 en su lugar')

## 2. Extraer y analizar fotogramas con Claude

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

def extraer_fotogramas(ruta_video: str, fps: float = 1.0) -> list[Path]:
    directorio = Path('/tmp/frames')
    directorio.mkdir(exist_ok=True)
    # Limpiar frames anteriores
    for f in directorio.glob('frame_*.jpg'):
        f.unlink()

    subprocess.run([
        'ffmpeg', '-i', ruta_video,
        '-vf', f'fps={fps}',
        '-q:v', '3',
        str(directorio / 'frame_%04d.jpg'),
        '-y', '-loglevel', 'quiet'
    ], check=True)

    return sorted(directorio.glob('frame_*.jpg'))

try:
    fotogramas = extraer_fotogramas(VIDEO_PRUEBA, fps=1.0)
    print(f'Extraídos {len(fotogramas)} fotogramas')

    # Mostrar los primeros 3
    n = min(3, len(fotogramas))
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 3))
    if n == 1:
        axes = [axes]
    for i, (ax, fotograma) in enumerate(zip(axes, fotogramas[:n])):
        img = mpimg.imread(str(fotograma))
        ax.imshow(img)
        ax.set_title(f'Segundo {i + 1}')
        ax.axis('off')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'Error: {e}')

In [ ]:
def analizar_fotogramas_con_claude(fotogramas: list[Path], pregunta: str, max_frames: int = 5) -> str:
    """Envía múltiples fotogramas a Claude Vision para análisis."""
    # Seleccionar fotogramas distribuidos uniformemente
    if len(fotogramas) > max_frames:
        indices = [int(i * len(fotogramas) / max_frames) for i in range(max_frames)]
        fotogramas_sel = [fotogramas[i] for i in indices]
    else:
        fotogramas_sel = fotogramas

    contenido = []
    for i, fotograma in enumerate(fotogramas_sel):
        with open(fotograma, 'rb') as f:
            datos_b64 = base64.standard_b64encode(f.read()).decode()
        contenido.append({'type': 'text', 'text': f'Fotograma {i + 1}:'})
        contenido.append({
            'type': 'image',
            'source': {'type': 'base64', 'media_type': 'image/jpeg', 'data': datos_b64}
        })

    contenido.append({'type': 'text', 'text': f'\n{pregunta}'})

    resp = client.messages.create(
        model='claude-sonnet-4-6',  # Sonnet para mejor análisis visual
        max_tokens=500,
        messages=[{'role': 'user', 'content': contenido}]
    )
    return resp.content[0].text

try:
    analisis = analizar_fotogramas_con_claude(
        fotogramas,
        'Describe qué ves en estos fotogramas. ¿Hay texto, personas u objetos notables?'
    )
    print('Análisis de Claude Vision:')
    print(analisis)
except Exception as e:
    print(f'Error en análisis: {e}')

## 3. Transcripción de audio de video

In [ ]:
def extraer_audio(ruta_video: str, ruta_salida: str = '/tmp/audio_extraido.mp3') -> str:
    subprocess.run([
        'ffmpeg', '-i', ruta_video,
        '-q:a', '0', '-map', 'a',
        ruta_salida, '-y', '-loglevel', 'quiet'
    ], check=True)
    return ruta_salida

def transcribir_con_whisper(ruta_audio: str) -> dict:
    from openai import OpenAI
    oai = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

    with open(ruta_audio, 'rb') as f:
        transcripcion = oai.audio.transcriptions.create(
            model='whisper-1',
            file=f,
            language='es',
            response_format='verbose_json',
        )
    return {
        'texto': transcripcion.text,
        'duracion_s': transcripcion.duration,
        'segmentos': [{'inicio': s.start, 'fin': s.end, 'texto': s.text} for s in (transcripcion.segments or [])],
    }

# Demo con video de prueba (el video generado no tiene audio, solo sirve como estructura)
print('Pipeline de transcripción:')
print('1. extraer_audio(video.mp4) → audio.mp3')
print('2. transcribir_con_whisper(audio.mp3) → {texto, duracion, segmentos}')
print('3. resumir_transcripcion(texto) → {puntos_clave, accion_items, resumen}')

# Simular transcripción para demo (sin audio real)
transcripcion_simulada = {
    'texto': 'Buenos días a todos. En esta reunión vamos a revisar el estado del proyecto IA. '
             'El primer punto es el análisis de costes. Hemos reducido un 30% el gasto en APIs '
             'usando prompt caching. El siguiente punto son los plazos de entrega.',
    'duracion_s': 45.2,
    'segmentos': [
        {'inicio': 0.0, 'fin': 8.0, 'texto': 'Buenos días a todos.'},
        {'inicio': 8.0, 'fin': 20.0, 'texto': 'En esta reunión vamos a revisar el estado del proyecto.'},
    ]
}
print(f'\nTranscripción simulada: {len(transcripcion_simulada["texto"].split())} palabras')

## 4. Resumen de reunión con Claude

In [ ]:
def resumir_reunion(transcripcion: str) -> dict:
    resp = client.messages.create(
        model=MODELO, max_tokens=600, temperature=0,
        messages=[{'role': 'user', 'content': f'''Analiza la transcripción de esta reunión. JSON:
{{
  "duracion_estimada_min": 0,
  "participantes_detectados": [],
  "temas_principales": [],
  "decisiones_tomadas": [],
  "accion_items": [{{
    "tarea": "...",
    "responsable": "...",
    "fecha_limite": "..."
  }}],
  "resumen_ejecutivo": "..."
}}

TRANSCRIPCIÓN:
{transcripcion}'''}]
    )
    texto = resp.content[0].text
    if '```' in texto:
        texto = texto.split('```')[1].replace('json', '').strip()
    return json.loads(texto)

resumen = resumir_reunion(transcripcion_simulada['texto'])
print('RESUMEN DE REUNIÓN:')
print(f'Duración estimada: {resumen.get("duracion_estimada_min", "?")} min')
print(f'\nTemas: {resumen.get("temas_principales", [])}')
print(f'\nDecisiones: {resumen.get("decisiones_tomadas", [])}')
print(f'\nAction Items:')
for item in resumen.get('accion_items', []):
    print(f'  • {item.get("tarea", "?")} → {item.get("responsable", "?")} ({item.get("fecha_limite", "?")})')
print(f'\nResumen: {resumen.get("resumen_ejecutivo", "?")}')

## 5. Generación de subtítulos SRT

In [ ]:
def segmentos_a_srt(segmentos: list[dict]) -> str:
    def tiempo_srt(segundos: float) -> str:
        h, rem = divmod(int(segundos), 3600)
        m, s = divmod(rem, 60)
        ms = int((segundos % 1) * 1000)
        return f'{h:02d}:{m:02d}:{s:02d},{ms:03d}'

    lineas = []
    for i, seg in enumerate(segmentos, 1):
        lineas.append(f'{i}')
        lineas.append(f'{tiempo_srt(seg["inicio"])} --> {tiempo_srt(seg["fin"])}')
        lineas.append(seg['texto'].strip())
        lineas.append('')
    return '\n'.join(lineas)

srt = segmentos_a_srt(transcripcion_simulada['segmentos'])
print('Archivo SRT generado:')
print(srt)

# Guardar
with open('/tmp/subtitulos.srt', 'w', encoding='utf-8') as f:
    f.write(srt)
print('\nGuardado en /tmp/subtitulos.srt')

## 6. Calculadora de costes para pipelines de video

In [ ]:
def calcular_coste_pipeline_video(
    duracion_min: float,
    n_videos: int = 1,
) -> dict:
    # Whisper: $0.006/minuto
    coste_whisper = duracion_min * 0.006 * n_videos

    # Claude Haiku para resumen: ~500 tokens entrada + 300 salida por video
    coste_claude = (500 * 0.80 + 300 * 4.00) / 1_000_000 * n_videos

    # Claude Sonnet Vision para fotogramas: ~5 imágenes × 500 tokens/imagen
    coste_vision = (5 * 500 * 3.00 + 300 * 15.00) / 1_000_000 * n_videos

    # Gemini 1.5 Pro para análisis completo de video
    # ~$0.00265/minuto de video con Gemini 1.5 Pro
    coste_gemini = duracion_min * 0.00265 * n_videos

    return {
        'duracion_total_min': duracion_min * n_videos,
        'whisper_transcripcion': coste_whisper,
        'claude_haiku_resumen': coste_claude,
        'claude_sonnet_vision': coste_vision,
        'gemini_pro_video': coste_gemini,
        'pipeline_completo_claude': coste_whisper + coste_claude + coste_vision,
        'pipeline_completo_gemini': coste_whisper + coste_gemini,
    }

escenarios = [
    {'desc': 'Reunión corta (30 min)', 'duracion': 30, 'n': 1},
    {'desc': 'Curso online (60 min × 10 lecciones)', 'duracion': 60, 'n': 10},
    {'desc': 'Archivo de llamadas (20 min × 100/mes)', 'duracion': 20, 'n': 100},
]

print(f'{"Escenario":<45} {"Pipeline Claude":>16} {"Pipeline Gemini":>16}')
print('-' * 80)
for e in escenarios:
    costes = calcular_coste_pipeline_video(e['duracion'], e['n'])
    print(f'{e["desc"]:<45} ${costes["pipeline_completo_claude"]:>14.3f} ${costes["pipeline_completo_gemini"]:>14.3f}')